# UCI HAR → Seq2Seq Transformer Data Split


- **Subject** = sequence instance, analogous to a wellbore.
- **UCI HAR window index** = sequence axis, analogous to depth.
- **Activity label at each window** = per-position seq2seq target.
- **Activity transitions** = optional edge/top-like targets, analogous to geological tops.

## Main outputs

Saved in `Transformer/data/` (next to this `src/` notebook):

| File | Content |
|---|---|
| `train.npz` | `X`, `y_activity`, `y_transition`, `y_edge`, `valid_mask`, `subject_ids` |
| `val.npz` | same schema |
| `test.npz` | same schema |
| `warm_state.json` | metadata needed to configure the transformer |

These are the files consumed by `Transformer/uci_har_seq2seq.ipynb`.

## Shapes

The output follows the same orientation as the well-log transformer:

```text
X           : (n_subjects, n_channels, seq_len)
y_activity  : (n_subjects, seq_len)        # 0–5 activity at each position
y_transition: (n_subjects, seq_len)        # 0=None, 1–6 entering activity, IGNORE_INDEX for padding
y_edge      : (n_subjects, seq_len, 6)     # marker-specific transition targets
valid_mask  : (n_subjects, seq_len)        # True for real positions, False for padding
```

By default, each UCI HAR window is converted into compact per-channel statistics. This keeps `n_channels` manageable for PatchTST. You can switch to `FEATURE_MODE = "flatten"` if you want to preserve every raw value from each 9×128 UCI HAR window.

In [ ]:
import os
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

IGNORE_INDEX = -100
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# =============================================================================
# 1. Paths and configuration
# =============================================================================

# Resolve paths relative to the repository root so the notebook runs from any
# working directory. The repo root is found by walking up for `setup.py` + `geoxts/`.
# This notebook lives at: datasets/medical/Transformer/src/
# Raw data lives at      : datasets/medical/UCI HAR Dataset/
# Splits are written to  : datasets/medical/Transformer/data/  (next to src/)


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "setup.py").exists() and (parent / "geoxts").is_dir():
            return parent
    return start


NB_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NB_DIR)

# UCI_HAR_DIR env var takes priority (useful on a VM where data lives outside the repo).
_env_har = os.environ.get("UCI_HAR_DIR")
HAR_CANDIDATES = [
    Path(_env_har) if _env_har else None,
    REPO_ROOT / "datasets" / "medical" / "UCI HAR Dataset",
    NB_DIR.parent.parent / "UCI HAR Dataset",          # Transformer/src -> medical/UCI HAR Dataset
    Path("UCI HAR Dataset"),
]
HAR_ROOT = next((p for p in HAR_CANDIDATES if p and p.exists()), HAR_CANDIDATES[1])

# Write the seq2seq splits into Transformer/data/ (one level up from this src/ notebook).
SAVE_DIR = NB_DIR.parent / "data"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

if not HAR_ROOT.exists():
    raise FileNotFoundError(
        f"UCI HAR Dataset not found at: {HAR_ROOT.resolve()}\n"
        "Download/unzip it into datasets/medical/ (see datasets/medical/README.md), "
        "or set the UCI_HAR_DIR environment variable to its path."
    )

# UCI HAR activity labels are stored as 1–6 in the raw files.
# We remap them to 0–5 for activity segmentation.
ACTIVITY_LABELS = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]

# Top-like transition classes: 0=None, 1–6=activity being entered.
PRED_COLUMN = ["None"] + ACTIVITY_LABELS

CHANNELS = [
    "body_acc_x",  "body_acc_y",  "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z",
]

WINDOW_LEN = 128
N_RAW_CHANNELS = len(CHANNELS)
N_ACTIVITIES = len(ACTIVITY_LABELS)
N_TRANSITION_CLASSES = len(PRED_COLUMN)

# UCI HAR train subjects are fixed by the original dataset.
# We keep the original test set untouched and carve out validation subjects from train.
VAL_SUBJECTS = {22, 25, 27, 29}

# Feature mode:
# - "stats"   : recommended for PatchTST; compact channels from each 9x128 window.
# - "flatten" : no compression; each original window becomes 9*128=1152 features per sequence step.
FEATURE_MODE = "stats"

# Transition target parameters.
# y_transition marks only exact change points. y_edge creates a Gaussian-like target around them.
EDGE_HALF_WIDTH_WINDOWS = 2
EDGE_SIGMA_WINDOWS = 1.0

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"HAR_ROOT : {HAR_ROOT.resolve()}")
print(f"SAVE_DIR : {SAVE_DIR.resolve()}")
print(f"Feature mode: {FEATURE_MODE}")

In [27]:
# =============================================================================
# 2. Load UCI HAR raw inertial signal windows
# =============================================================================

def load_raw_split(split: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Load one UCI HAR split in temporal order.

    Returns:
        X_raw    : (N, 9, 128) float32
        y        : (N,) int64, activity labels remapped from 1–6 to 0–5
        subjects : (N,) int64, subject id for each UCI HAR window
    """
    split_dir = HAR_ROOT / split
    sig_dir = split_dir / "Inertial Signals"

    arrays = []
    for ch in CHANNELS:
        path = sig_dir / f"{ch}_{split}.txt"
        if not path.exists():
            raise FileNotFoundError(f"Missing inertial signal file: {path}")
        arrays.append(np.loadtxt(path, dtype=np.float32))

    X_raw = np.stack(arrays, axis=1).astype(np.float32)  # (N, 9, 128)
    y_raw = np.loadtxt(split_dir / f"y_{split}.txt", dtype=np.int64)
    subjects = np.loadtxt(split_dir / f"subject_{split}.txt", dtype=np.int64)

    y = y_raw - 1

    assert X_raw.ndim == 3 and X_raw.shape[1:] == (N_RAW_CHANNELS, WINDOW_LEN)
    assert len(X_raw) == len(y) == len(subjects)
    assert y.min() >= 0 and y.max() < N_ACTIVITIES

    return X_raw, y.astype(np.int64), subjects.astype(np.int64)


X_train_raw_all, y_train_raw_all, subj_train_all = load_raw_split("train")
X_test_raw, y_test_raw, subj_test = load_raw_split("test")

print(f"Raw train: X={X_train_raw_all.shape}, y={y_train_raw_all.shape}, subjects={sorted(np.unique(subj_train_all).tolist())}")
print(f"Raw test : X={X_test_raw.shape}, y={y_test_raw.shape}, subjects={sorted(np.unique(subj_test).tolist())}")

Raw train: X=(7352, 9, 128), y=(7352,), subjects=[1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 17, 19, 21, 22, 23, 25, 26, 27, 28, 29, 30]
Raw test : X=(2947, 9, 128), y=(2947,), subjects=[2, 4, 9, 10, 12, 13, 18, 20, 24]


In [28]:
# =============================================================================
# 3. Subject-based train/validation split
# =============================================================================

val_mask = np.isin(subj_train_all, sorted(VAL_SUBJECTS))
train_mask = ~val_mask

X_train_raw = X_train_raw_all[train_mask]
y_train_raw = y_train_raw_all[train_mask]
subj_train = subj_train_all[train_mask]

X_val_raw = X_train_raw_all[val_mask]
y_val_raw = y_train_raw_all[val_mask]
subj_val = subj_train_all[val_mask]

print(f"Train windows: {X_train_raw.shape}, subjects={sorted(np.unique(subj_train).tolist())}")
print(f"Val windows  : {X_val_raw.shape}, subjects={sorted(np.unique(subj_val).tolist())}")
print(f"Test windows : {X_test_raw.shape}, subjects={sorted(np.unique(subj_test).tolist())}")

Train windows: (5902, 9, 128), subjects=[1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 17, 19, 21, 23, 26, 28, 30]
Val windows  : (1450, 9, 128), subjects=[22, 25, 27, 29]
Test windows : (2947, 9, 128), subjects=[2, 4, 9, 10, 12, 13, 18, 20, 24]


In [29]:
# =============================================================================
# 4. Convert each UCI HAR window into per-step features
# =============================================================================

def window_to_features(X_raw: np.ndarray, mode: str = FEATURE_MODE) -> Tuple[np.ndarray, List[str]]:
    """Convert raw UCI HAR windows into features used as transformer channels.

    Input:
        X_raw: (N, 9, 128)

    Output:
        X_feat: (N, n_features)

    Interpretation:
        Each original UCI HAR window becomes one position on the sequence axis.
        The feature dimension becomes the channel axis of the transformer.
    """
    if mode == "flatten":
        names = [f"{ch}_t{t}" for ch in CHANNELS for t in range(WINDOW_LEN)]
        return X_raw.reshape(X_raw.shape[0], -1).astype(np.float32), names

    if mode == "stats":
        stats = []
        names = []
        stat_fns = {
            "mean": lambda a: a.mean(axis=2),
            "std":  lambda a: a.std(axis=2),
            "min":  lambda a: a.min(axis=2),
            "max":  lambda a: a.max(axis=2),
            "first": lambda a: a[:, :, 0],
            "last":  lambda a: a[:, :, -1],
        }
        for stat_name, fn in stat_fns.items():
            stats.append(fn(X_raw))  # each is (N, 9)
            names.extend([f"{ch}_{stat_name}" for ch in CHANNELS])
        return np.concatenate(stats, axis=1).astype(np.float32), names

    raise ValueError(f"Unknown FEATURE_MODE: {mode}. Use 'stats' or 'flatten'.")


X_train_feat, FEATURE_NAMES = window_to_features(X_train_raw, FEATURE_MODE)
X_val_feat, _ = window_to_features(X_val_raw, FEATURE_MODE)
X_test_feat, _ = window_to_features(X_test_raw, FEATURE_MODE)

print(f"Feature matrix shapes:")
print(f"  train: {X_train_feat.shape}")
print(f"  val  : {X_val_feat.shape}")
print(f"  test : {X_test_feat.shape}")
print(f"n_channels for transformer = {len(FEATURE_NAMES)}")
print(FEATURE_NAMES[:10], "...")

Feature matrix shapes:
  train: (5902, 54)
  val  : (1450, 54)
  test : (2947, 54)
n_channels for transformer = 54
['body_acc_x_mean', 'body_acc_y_mean', 'body_acc_z_mean', 'body_gyro_x_mean', 'body_gyro_y_mean', 'body_gyro_z_mean', 'total_acc_x_mean', 'total_acc_y_mean', 'total_acc_z_mean', 'body_acc_x_std'] ...


In [30]:
# =============================================================================
# 5. Standardize features using training-set statistics only
# =============================================================================

feat_mean = X_train_feat.mean(axis=0, keepdims=True).astype(np.float32)
feat_std = X_train_feat.std(axis=0, keepdims=True).astype(np.float32)
feat_std = np.where(feat_std < 1e-8, 1.0, feat_std).astype(np.float32)

X_train_feat = ((X_train_feat - feat_mean) / feat_std).astype(np.float32)
X_val_feat = ((X_val_feat - feat_mean) / feat_std).astype(np.float32)
X_test_feat = ((X_test_feat - feat_mean) / feat_std).astype(np.float32)

print("Standardization fitted on train and applied to val/test.")
print(f"Train standardized mean={X_train_feat.mean():+.4f}, std={X_train_feat.std():.4f}")

Standardization fitted on train and applied to val/test.
Train standardized mean=-0.0000, std=1.0000


In [31]:
# =============================================================================
# 6. Build seq2seq targets per subject
# =============================================================================

def build_transition_targets(
    y_activity: np.ndarray,
    max_len: int,
    valid_len: int,
    half_width: int = EDGE_HALF_WIDTH_WINDOWS,
    sigma: float = EDGE_SIGMA_WINDOWS,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create top-like transition targets for one subject sequence.

    y_activity uses 0–5 activities at every real position.

    y_transition uses:
        0       = None / no activity transition
        1–6     = new activity being entered at transition position
        -100    = padding

    y_edge has one curve per activity class. At a transition into activity k,
    y_edge[:, k] receives a Gaussian-like peak around that transition.
    Padded positions are -1 and real non-transition positions are 0.
    """
    y_transition = np.full(max_len, IGNORE_INDEX, dtype=np.int64)
    y_transition[:valid_len] = 0

    y_edge = np.full((max_len, N_ACTIVITIES), -1.0, dtype=np.float32)
    y_edge[:valid_len, :] = 0.0

    if valid_len <= 1:
        return y_transition, y_edge

    transition_idx = np.where(np.diff(y_activity[:valid_len]) != 0)[0] + 1

    positions = np.arange(max_len)
    for t in transition_idx:
        entered_activity = int(y_activity[t])  # 0–5
        y_transition[t] = entered_activity + 1  # 1–6, 0 reserved for None

        local = (positions < valid_len) & (np.abs(positions - t) <= half_width)
        y_edge[local, entered_activity] = np.maximum(
            y_edge[local, entered_activity],
            np.exp(-0.5 * ((positions[local] - t) / sigma) ** 2),
        )

    return y_transition, y_edge


def subject_lengths(subjects: np.ndarray) -> Dict[int, int]:
    return {int(s): int((subjects == s).sum()) for s in np.unique(subjects)}


all_lengths = {}
for split_name, subjects in [("train", subj_train), ("val", subj_val), ("test", subj_test)]:
    all_lengths[split_name] = subject_lengths(subjects)
    print(split_name, all_lengths[split_name])

MAX_SEQ_LEN = max(max(d.values()) for d in all_lengths.values())
print(f"MAX_SEQ_LEN across train/val/test subjects = {MAX_SEQ_LEN}")

train {1: 347, 3: 341, 5: 302, 6: 325, 7: 308, 8: 281, 11: 316, 14: 323, 15: 328, 16: 366, 17: 368, 19: 360, 21: 408, 23: 372, 26: 392, 28: 382, 30: 383}
val {22: 321, 25: 409, 27: 376, 29: 344}
test {2: 302, 4: 317, 9: 288, 10: 294, 12: 320, 13: 327, 18: 364, 20: 354, 24: 381}
MAX_SEQ_LEN across train/val/test subjects = 409


In [32]:
def make_subject_sequences(
    X_feat: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    max_seq_len: int = MAX_SEQ_LEN,
) -> Dict[str, np.ndarray]:
    """Pack window-level features into padded subject-level sequences.

    Returns a dictionary with:
        X            : (n_subjects, n_features, max_seq_len)
        y_activity   : (n_subjects, max_seq_len), 0–5 real labels, IGNORE_INDEX padding
        y_transition : (n_subjects, max_seq_len), 0=None, 1–6 entering activity, IGNORE_INDEX padding
        y_edge       : (n_subjects, max_seq_len, 6), -1 padding/ignored
        valid_mask   : (n_subjects, max_seq_len)
        subject_ids  : (n_subjects,)
        lengths      : (n_subjects,)
    """
    unique_subjects = sorted(np.unique(subjects).astype(int).tolist())
    n_subjects = len(unique_subjects)
    n_features = X_feat.shape[1]

    X_seq = np.zeros((n_subjects, n_features, max_seq_len), dtype=np.float32)
    y_activity = np.full((n_subjects, max_seq_len), IGNORE_INDEX, dtype=np.int64)
    y_transition = np.full((n_subjects, max_seq_len), IGNORE_INDEX, dtype=np.int64)
    y_edge = np.full((n_subjects, max_seq_len, N_ACTIVITIES), -1.0, dtype=np.float32)
    valid_mask = np.zeros((n_subjects, max_seq_len), dtype=bool)
    lengths = np.zeros(n_subjects, dtype=np.int64)

    for i, subj in enumerate(unique_subjects):
        idx = np.where(subjects == subj)[0]
        # UCI files are already temporally ordered within subject; keep that order.
        L = len(idx)
        if L > max_seq_len:
            raise ValueError(f"Subject {subj} length {L} exceeds max_seq_len={max_seq_len}.")

        X_s = X_feat[idx]       # (L, n_features)
        y_s = y[idx].astype(np.int64)  # (L,)

        X_seq[i, :, :L] = X_s.T
        y_activity[i, :L] = y_s
        valid_mask[i, :L] = True
        lengths[i] = L

        yt, ye = build_transition_targets(y_s, max_seq_len, L)
        y_transition[i] = yt
        y_edge[i] = ye

    return {
        "X": X_seq,
        "y_activity": y_activity,
        "y_transition": y_transition,
        "y_edge": y_edge,
        "valid_mask": valid_mask,
        "subject_ids": np.array(unique_subjects, dtype=np.int64),
        "lengths": lengths,
    }


train_data = make_subject_sequences(X_train_feat, y_train_raw, subj_train)
val_data = make_subject_sequences(X_val_feat, y_val_raw, subj_val)
test_data = make_subject_sequences(X_test_feat, y_test_raw, subj_test)

for name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    print(f"\n{name}")
    print(f"  X           : {data['X'].shape}")
    print(f"  y_activity  : {data['y_activity'].shape}")
    print(f"  y_transition: {data['y_transition'].shape}")
    print(f"  y_edge      : {data['y_edge'].shape}")
    print(f"  lengths     : min={data['lengths'].min()}, max={data['lengths'].max()}, mean={data['lengths'].mean():.1f}")


train
  X           : (17, 54, 409)
  y_activity  : (17, 409)
  y_transition: (17, 409)
  y_edge      : (17, 409, 6)
  lengths     : min=281, max=408, mean=347.2

val
  X           : (4, 54, 409)
  y_activity  : (4, 409)
  y_transition: (4, 409)
  y_edge      : (4, 409, 6)
  lengths     : min=321, max=409, mean=362.5

test
  X           : (9, 54, 409)
  y_activity  : (9, 409)
  y_transition: (9, 409)
  y_edge      : (9, 409, 6)
  lengths     : min=288, max=381, mean=327.4


In [33]:
# =============================================================================
# 7. Diagnostics
# =============================================================================

def print_activity_distribution(name: str, y_activity: np.ndarray):
    valid = y_activity[y_activity != IGNORE_INDEX]
    classes, counts = np.unique(valid, return_counts=True)
    total = len(valid)
    print(f"\n{name} activity labels ({total} valid positions):")
    for c, n in zip(classes, counts):
        print(f"  [{c}] {ACTIVITY_LABELS[int(c)]:<20} {n:>6} ({100*n/total:5.1f}%)")


def print_transition_distribution(name: str, y_transition: np.ndarray):
    valid = y_transition[y_transition != IGNORE_INDEX]
    classes, counts = np.unique(valid, return_counts=True)
    total = len(valid)
    print(f"\n{name} transition labels ({total} valid positions):")
    for c, n in zip(classes, counts):
        print(f"  [{c}] {PRED_COLUMN[int(c)]:<20} {n:>6} ({100*n/total:5.1f}%)")

for name, data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    print_activity_distribution(name, data["y_activity"])
    print_transition_distribution(name, data["y_transition"])

# Check exact transition count by split.
for name, data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    n_transitions = int(((data["y_transition"] > 0) & (data["y_transition"] != IGNORE_INDEX)).sum())
    print(f"{name:<5} transitions: {n_transitions}")


Train activity labels (5902 valid positions):
  [0] WALKING                 996 ( 16.9%)
  [1] WALKING_UPSTAIRS        866 ( 14.7%)
  [2] WALKING_DOWNSTAIRS      800 ( 13.6%)
  [3] SITTING                1029 ( 17.4%)
  [4] STANDING               1092 ( 18.5%)
  [5] LAYING                 1119 ( 19.0%)

Train transition labels (5902 valid positions):
  [0] None                   5689 ( 96.4%)
  [1] WALKING                  34 (  0.6%)
  [2] WALKING_UPSTAIRS         45 (  0.8%)
  [3] WALKING_DOWNSTAIRS       47 (  0.8%)
  [4] SITTING                  35 (  0.6%)
  [5] STANDING                 17 (  0.3%)
  [6] LAYING                   35 (  0.6%)

Val activity labels (1450 valid positions):
  [0] WALKING                 230 ( 15.9%)
  [1] WALKING_UPSTAIRS        207 ( 14.3%)
  [2] WALKING_DOWNSTAIRS      186 ( 12.8%)
  [3] SITTING                 257 ( 17.7%)
  [4] STANDING                282 ( 19.4%)
  [5] LAYING                  288 ( 19.9%)

Val transition labels (1450 valid positio

In [34]:
# =============================================================================
# 8. Save NPZ files and warm_state metadata
# =============================================================================

for split_name, data in [("train", train_data), ("val", val_data), ("test", test_data)]:
    np.savez_compressed(
        SAVE_DIR / f"{split_name}.npz",
        X=data["X"].astype(np.float32),
        y_activity=data["y_activity"].astype(np.int64),
        y_transition=data["y_transition"].astype(np.int64),
        y_edge=data["y_edge"].astype(np.float32),
        valid_mask=data["valid_mask"].astype(bool),
        subject_ids=data["subject_ids"].astype(np.int64),
        lengths=data["lengths"].astype(np.int64),
    )

all_subjects = sorted(
    set(train_data["subject_ids"].tolist()) |
    set(val_data["subject_ids"].tolist()) |
    set(test_data["subject_ids"].tolist())
)
subject2id = {str(s): i for i, s in enumerate(all_subjects)}

warm_state = {
    "dataset_type": "uci_har_seq2seq_transformer",
    "HAR_ROOT": str(HAR_ROOT),
    "SAVE_DIR": str(SAVE_DIR),
    "IGNORE_INDEX": IGNORE_INDEX,
    "FEATURE_MODE": FEATURE_MODE,
    "WINDOW_LEN": WINDOW_LEN,
    "RAW_CHANNELS": CHANNELS,
    "FEATURES_GEOX": FEATURE_NAMES,
    "N_CHANNELS": len(FEATURE_NAMES),
    "MAX_SEQ_LEN": int(MAX_SEQ_LEN),
    "ACTIVITY_LABELS": ACTIVITY_LABELS,
    "PRED_COLUMN": PRED_COLUMN,
    "N_ACTIVITY_CLASSES": N_ACTIVITIES,
    "N_TRANSITION_CLASSES": N_TRANSITION_CLASSES,
    "EDGE_HALF_WIDTH_WINDOWS": EDGE_HALF_WIDTH_WINDOWS,
    "EDGE_SIGMA_WINDOWS": EDGE_SIGMA_WINDOWS,
    "WELL_COL": "subject_id",
    "SEQ_AXIS": "uci_har_window_index",
    "train_wells": train_data["subject_ids"].astype(int).tolist(),
    "val_wells": val_data["subject_ids"].astype(int).tolist(),
    "test_wells": test_data["subject_ids"].astype(int).tolist(),
    "well2id": subject2id,
    "feature_mean": feat_mean.flatten().astype(float).tolist(),
    "feature_std": feat_std.flatten().astype(float).tolist(),
    "notes": {
        "X_shape": "(n_subjects, n_channels, seq_len)",
        "y_activity": "Per-position activity segmentation target: 0-5, IGNORE_INDEX for padding.",
        "y_transition": "Top-like transition target: 0=None, 1-6=activity being entered, IGNORE_INDEX for padding.",
        "y_edge": "Marker-specific transition curves with shape (n_subjects, seq_len, 6).",
    },
}

with open(SAVE_DIR / "warm_state.json", "w") as f:
    json.dump(warm_state, f, indent=2)

print(f"Saved seq2seq transformer data to: {SAVE_DIR.resolve()}")
for split_name in ["train", "val", "test"]:
    print(f"  {split_name}.npz")
print("  warm_state.json")

Saved seq2seq transformer data to: /Users/enorabarbier/Documents/1.M2 Courses/CRP/CRP_SLB/datasets/medical/Transformer/har_seq2seq_transformer
  train.npz
  val.npz
  test.npz
  warm_state.json


In [35]:
# =============================================================================
# 9. Reload and sanity-check saved files
# =============================================================================

for split_name in ["train", "val", "test"]:
    path = SAVE_DIR / f"{split_name}.npz"
    d = np.load(path)

    X = d["X"]
    y_activity = d["y_activity"]
    y_transition = d["y_transition"]
    y_edge = d["y_edge"]
    valid_mask = d["valid_mask"]
    subject_ids = d["subject_ids"]

    assert X.ndim == 3, X.shape
    assert X.shape[1] == len(FEATURE_NAMES), X.shape
    assert X.shape[2] == MAX_SEQ_LEN, X.shape
    assert y_activity.shape == (X.shape[0], X.shape[2])
    assert y_transition.shape == (X.shape[0], X.shape[2])
    assert y_edge.shape == (X.shape[0], X.shape[2], N_ACTIVITIES)
    assert valid_mask.shape == y_activity.shape
    assert len(subject_ids) == X.shape[0]

    valid_acts = y_activity[y_activity != IGNORE_INDEX]
    valid_trans = y_transition[y_transition != IGNORE_INDEX]
    assert valid_acts.min() >= 0 and valid_acts.max() <= 5
    assert valid_trans.min() >= 0 and valid_trans.max() <= 6

    print(
        f"{split_name:<5} | X={X.shape} | y_activity={y_activity.shape} | "
        f"y_transition={y_transition.shape} | y_edge={y_edge.shape} | subjects={subject_ids.tolist()}"
    )

with open(SAVE_DIR / "warm_state.json") as f:
    ws = json.load(f)

assert ws["dataset_type"] == "uci_har_seq2seq_transformer"
assert ws["N_CHANNELS"] == len(FEATURE_NAMES)
assert ws["MAX_SEQ_LEN"] == MAX_SEQ_LEN

print("\nAll sanity checks passed.")

train | X=(17, 54, 409) | y_activity=(17, 409) | y_transition=(17, 409) | y_edge=(17, 409, 6) | subjects=[1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 17, 19, 21, 23, 26, 28, 30]
val   | X=(4, 54, 409) | y_activity=(4, 409) | y_transition=(4, 409) | y_edge=(4, 409, 6) | subjects=[22, 25, 27, 29]
test  | X=(9, 54, 409) | y_activity=(9, 409) | y_transition=(9, 409) | y_edge=(9, 409, 6) | subjects=[2, 4, 9, 10, 12, 13, 18, 20, 24]

All sanity checks passed.


In [ ]:
# =============================================================================
# 10. Minimal PyTorch Dataset wrapper for transformer
# =============================================================================

class HARSeq2SeqDataset:
    """Minimal dataset wrapper.

    Use target="activity" for normal seq2seq activity segmentation:
        X: (channels, seq_len)
        y: (seq_len,) with labels 0–5

    Use target="transition" for North-Sea-like top/edge learning:
        X:      (channels, seq_len)
        y:      (seq_len,) with labels 0=None, 1–6 transition class
        y_edge: (seq_len, 6)
    """
    def __init__(self, npz_path: str | Path, target: str = "activity"):
        self.data = np.load(npz_path)
        self.X = self.data["X"].astype(np.float32)
        self.valid_mask = self.data["valid_mask"].astype(bool)
        self.subject_ids = self.data["subject_ids"].astype(np.int64)
        self.target = target

        if target == "activity":
            self.y = self.data["y_activity"].astype(np.int64)
            self.y_edge = None
        elif target == "transition":
            self.y = self.data["y_transition"].astype(np.int64)
            self.y_edge = self.data["y_edge"].astype(np.float32)
        else:
            raise ValueError("target must be 'activity' or 'transition'")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.target == "transition":
            return self.X[idx], self.y[idx], self.y_edge[idx], self.valid_mask[idx], self.subject_ids[idx]
        return self.X[idx], self.y[idx], self.valid_mask[idx], self.subject_ids[idx]


train_ds_activity = HARSeq2SeqDataset(SAVE_DIR / "train.npz", target="activity")
train_ds_transition = HARSeq2SeqDataset(SAVE_DIR / "train.npz", target="transition")

print("Activity target sample:")
X0, y0, mask0, sid0 = train_ds_activity[0]
print(f"  subject={sid0}, X={X0.shape}, y={y0.shape}, valid positions={mask0.sum()}")

print("Transition target sample:")
X0, y0, edge0, mask0, sid0 = train_ds_transition[0]
print(f"  subject={sid0}, X={X0.shape}, y={y0.shape}, y_edge={edge0.shape}, valid positions={mask0.sum()}")

Activity target sample:
  subject=1, X=(54, 409), y=(409,), valid positions=347
Transition target sample:
  subject=1, X=(54, 409), y=(409,), y_edge=(409, 6), valid positions=347
